# Wearable Sleep Measures and Clinical Outcomes

This notebook combines prepared sleep-derived wearable measurements with viral load derived from nasal swab PCR test, symptom evaluation from Modified Jackson Score, and participant demographics. It first displays descriptive daily trajectories and then tests pre/post-inoculation differences and associations with clinical outcomes using linear mixed-effects models.

## Import packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")

## Load and merge the data

The four input tables contain sleep-derived wearable measurements, PCR viral load in copies/mL, symptom scores, and participant information. Wearable and clinical tables are joined on `Participant_ID` and `Day`; demographic covariates are added only for adjusted models.

Since the clinical assessments (viral load and symptom scores) were performed after inoculation, missing values before the inoculation after the viral load merge are filled with 0.5, and all remaining missing values after the symptom-score merge are filled with 0. 

In [ ]:
# Load the demographics and test information data
test_info = pd.read_csv('test_information.csv')

# load final calculated wearable data during sleep
wearable_data = pd.read_csv('data/wearable_sleep.csv')

# load viral loads
viral_loads = pd.read_csv('data/viral_loads.csv')

# load symptom data
symptom_score = pd.read_csv('data/symptom_scores.csv')

# merge wearable data and clinical assessments together for statistical analysis
final_data = pd.merge(wearable_data, viral_loads, on=['Participant_ID', 'Day'], how='left').fillna(0.5)
final_df = pd.merge(final_data, symptom_score, on=['Participant_ID', 'Day'], how='left').fillna(0)

## Visualization: wearable measures and clinical assessments

Each panel shows daily mean wearable deviation, viral load, and symptom score for Symptomatic Infected participants on Days 0–7. Viral load is displayed on its original copies/mL scale. The HRV axis is inverted.

The three curves use separate y-axes; their visual overlap is not an estimate of association or a statistical test. These are group-level averages rather than individual trajectories. The models below use participant-level observations.

In [ ]:
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 7,
    'axes.labelsize': 7, 'axes.titlesize': 7,
    'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 6,
    'axes.linewidth': 0.6, 'lines.linewidth': 1.1, 'lines.markersize': 2.5,
    'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'savefig.dpi': 600
})

categories = ['Symptomatic and Infected']
reference_category = 'Symptomatic and Infected'
metric_labels = {
    'HR_Deviation': ('Heart Rate Deviation', 'HR Deviation (bpm)'),
    'Average_Breath_Deviation': ('Breathing Rate Deviation', 'RR Deviation (breaths/min)'),
    'Average_HRV_Deviation': ('Heart Rate Variability Deviation', 'HRV Deviation (ms)'),
    'Average_Temperature_Deviation': ('Temperature Deviation', 'Temperature Deviation (°C)')
}
viral_col = 'Viral Load (copies per mL)'
symptom_col = 'Symptom_Score'


def make_ylim(series, padding_ratio=0.1):
    """Add padding around finite values; use (0, 1) for an empty series."""
    values = pd.to_numeric(series, errors='coerce')
    values = values[np.isfinite(values)]
    if values.empty:
        return (0, 1)
    lower, upper = values.min(), values.max()
    padding = padding_ratio * (upper - lower if upper != lower else max(1, abs(upper)))
    return lower - padding, upper + padding


def plot_series(ax, data, column, label, color, marker, linestyle, limits):
    """Draw one curve and apply its y-axis label, color, and limits."""
    ax.plot(data['Day'], data[column], color=color, marker=marker,
            linestyle=linestyle, label=label)
    ax.set_ylabel(label, color=color)
    ax.tick_params(axis='y', labelcolor=color)
    ax.set_ylim(limits)


plot_columns = list(metric_labels) + [viral_col, symptom_col]
plot_df = final_df.copy()
plot_df[['Day'] + plot_columns] = plot_df[['Day'] + plot_columns].apply(
    pd.to_numeric, errors='coerce'
)
reference = plot_df.loc[
    (plot_df['INFSYM'] == reference_category) & (plot_df['Day'] >= 0) & (plot_df['Day'] < 8)
]
if reference.empty:
    raise ValueError(f'No data found for reference category: {reference_category}')
reference_means = reference.groupby('Day')[plot_columns].mean()
axis_limits = {column: make_ylim(reference_means[column]) for column in plot_columns}

for category in categories:
    subset = plot_df.loc[
        (plot_df['INFSYM'] == category) & (plot_df['Day'] >= 0) & (plot_df['Day'] < 8)
    ]
    if subset.empty:
        print(f'No data for {category}. Skipping.')
        continue
    # Calculate daily means once per category, rather than once per panel.
    grouped = subset.groupby('Day', as_index=False)[plot_columns].mean().sort_values('Day')
    fig, axes = plt.subplots(2, 2, figsize=(180 / 25.4, 160 / 25.4), sharex=False)

    for index, (ax, (metric, (title, label))) in enumerate(zip(axes.flat, metric_labels.items())):
        ax.text(0, 1.03, 'abcd'[index], transform=ax.transAxes,
                fontsize=8, fontweight='bold', va='bottom', clip_on=False)
        ax.text(0.07, 1.03, title, transform=ax.transAxes,
                fontsize=7, va='bottom', clip_on=False)
        ax.set_xlim(-0.2, 7.2)
        ax.set_xticks(range(8))
        ax.set_xlabel('Relative Day to Inoculation' if index >= 2 else '')
        ax.grid(False)

        limits = axis_limits[metric]
        if metric == 'Average_HRV_Deviation':
            limits = limits[::-1]
        plot_series(ax, grouped, metric, label, 'mediumorchid', 'o', '-', limits)

        viral_ax = ax.twinx()
        plot_series(viral_ax, grouped, viral_col, 'Viral load (copies/mL)',
                    'darkorange', 's', '--', axis_limits[viral_col])
        symptom_ax = ax.twinx()
        symptom_ax.spines['right'].set_position(('outward', 35))
        plot_series(symptom_ax, grouped, symptom_col, 'Symptom score',
                    'black', 'D', ':', axis_limits[symptom_col])

        # Show the clinical-outcome axes only in the right-hand column.
        if index % 2 == 0:
            for twin_ax in (viral_ax, symptom_ax):
                twin_ax.set_ylabel('')
                twin_ax.tick_params(axis='y', which='both', right=False, labelright=False)
                twin_ax.spines['right'].set_visible(False)
    plt.show()

# Statistical analysis

All analyses are restricted to symptomatic and infected participants with `Day > -2`. The same four deviation measures are used throughout: HR, breathing rate, HRV, and temperature. In particular, HR uses `HR_Deviation`, not `Normalized_HR_Deviation`.

Every model includes a participant-specific random intercept and uses the existing `MixedLM.fit()`. Adjusted models add SEX (M=1, F=0), AGE, and BMI. Each metric is modeled separately. Coefficient tests are two-sided Wald tests; `fit.tvalues` is labeled **Wald z** below to match the default mixed-model inference.

BH-FDR correction is performed separately for each analysis and adjustment status across four metrics. Day-specific comparisons are corrected across seven days within each metric. No additional correction is applied across all analyses combined.

## Shared functions and analysis settings

The functions below handle model fitting, demographic merging, and FDR correction. Each analysis section still explicitly defines its data and research question.

Missing model variables are removed before fitting so the model data and participant groups remain aligned. Failed fits remain in the results with an error message; an incomplete family does not receive FDR-adjusted p-values. Nonconverged fits are flagged, and their warnings remain visible. Numerical results are retained at full precision; formatting affects only display.

In [ ]:
metrics = [
    'HR_Deviation', 'Average_Breath_Deviation',
    'Average_HRV_Deviation', 'Average_Temperature_Deviation'
]
baseline_covariates = ['SEX', 'AGE', 'BMI']
selected_df = final_df.loc[
    (final_df['INFSYM'] == 'Symptomatic and Infected') & (final_df['Day'] > -2)
].copy().reset_index(drop=True)

covariate_data = test_info[['Participant ID'] + baseline_covariates].rename(
    columns={'Participant ID': 'Participant_ID'}
).copy()
covariate_data['SEX'] = covariate_data['SEX'].map({'M': 1, 'F': 0})


def add_covariates(data):
    """Attach baseline demographics without multiplying participant rows."""
    return data.merge(covariate_data, on='Participant_ID', how='left', validate='many_to_one')


def fit_mixed_model(data, outcome, predictor, covariates=()):
    """Fit a random-intercept model and return results for the predictor."""
    columns = ['Participant_ID', outcome, predictor] + list(covariates)
    model_data = data[columns].dropna().reset_index(drop=True)
    formula = f"{outcome} ~ " + ' + '.join([predictor] + list(covariates))
    result = {
        'Coefficient': np.nan, 'SE': np.nan,
        '95% CI Lower': np.nan, '95% CI Upper': np.nan,
    }
    try:
        if not np.isfinite(model_data[[outcome, predictor] + list(covariates)]).all().all():
            raise ValueError('Nonfinite model values; check input values and transformations.')
        if model_data[predictor].nunique() < 2:
            raise ValueError('The predictor has no variation in the analysis sample.')
        fit = smf.mixedlm(formula, model_data, groups=model_data['Participant_ID']).fit()
        lower, upper = fit.conf_int().loc[predictor]
        result.update({
            'Coefficient': fit.params[predictor], 
            'SE': fit.bse[predictor],
            '95% CI Lower': lower, '95% CI Upper': upper,
            'p-value': fit.pvalues[predictor],
        })
    except Exception as error:
        result['Error'] = str(error)
    return result


def adjust_fdr(results):
    """Apply BH-FDR to one complete family of planned tests."""
    results = results.copy()
    results['Adjusted p-value'] = np.nan
    results['Significant'] = pd.Series(pd.NA, index=results.index, dtype='boolean')
    if not results.empty and np.isfinite(results['p-value']).all():
        reject, adjusted, _, _ = multipletests(results['p-value'], alpha=0.05, method='fdr_bh')
        results['Adjusted p-value'] = adjusted
        results['Significant'] = reject
    else:
        print('FDR correction not applied: at least one test has no finite p-value. Review Error.')
    return results


def test_metrics(data, *, outcome=None, predictor=None, covariates=()):
    """Test each metric as an outcome (pre/post) or predictor (clinical association)."""
    if (outcome is None) == (predictor is None):
        raise ValueError('Specify either a shared outcome or a shared predictor.')
    rows = []
    for metric in metrics:
        row = fit_mixed_model(
            data, outcome=outcome or metric, predictor=predictor or metric,
            covariates=covariates
        )
        rows.append({'Metric': metric, **row})
    return adjust_fdr(pd.DataFrame(rows))


# Format for display only; stored coefficients and p-values remain numeric.
result_format = {
    'Coefficient': '{:.3f}', 'SE': '{:.3f}',
    '95% CI Lower': '{:.3f}', '95% CI Upper': '{:.3f}',
    'p-value': '{:.3e}', 'Adjusted p-value': '{:.3e}'
}

## 1. Test whether wearable deviations differ before and after inoculation

**Question:** Does the mean wearable deviation change from baseline to post-inoculation?

Pre-inoculation (baseline): Day < 0. Post-inoculation: Day >= 1. 
Note: the data are from sleep, which means the Day = 1 correspond to the data starting from the night on Day 0, which is after inoculation. 
First average each metric within participant and exposure period, then fit `wearable deviation ~ exposure_period`. 
The coefficient is the estimated post - pre (baseline) difference. BH-FDR is applied across four metrics.

In [ ]:
selected_df['exposure_period'] = (selected_df['Day'] >= 1).astype(int)
analysis_exposure = (
    selected_df[['Participant_ID', 'exposure_period'] + metrics]
    .groupby(['Participant_ID', 'exposure_period'])
    .mean().dropna().reset_index()
)

exposure_results = test_metrics(analysis_exposure, predictor='exposure_period')
display(exposure_results.style.format(result_format, na_rep='—'))

## 1.1 Test the pre/post difference after adjusting baseline covariants

**Question:** Does the pre/post difference remain after adjusting for SEX, AGE, and BMI?

Fit `wearable deviation ~ exposure_period + SEX + AGE + BMI`. Use the same participant-period averages as previous step. Explicitly remove missing covariates so model rows and participant groups stay aligned. BH-FDR is applied across four metrics.

In [ ]:
exposure_adjusted_results = test_metrics(
    add_covariates(analysis_exposure), predictor='exposure_period',
    covariates=baseline_covariates
)
display(exposure_adjusted_results.style.format(result_format, na_rep='—'))

## 2. Test each post-exposure day against baseline

**Question:** On each of post-inoculation day (Days 1–7), does the wearable metric differ from baseline (Day < 0)?

Fit `value ~ period`, where period=1 is the target day and period=0 is baseline. Keep individual daily baseline rows, as in the original code; do not average them first. Apply BH-FDR separately to the seven comparisons for each metric. Report all days rather than selecting only the most significant day.

In [ ]:
daily_tables = []
for metric in metrics:
    rows = []
    for day in range(1, 8):
        day_data = selected_df.loc[
            (selected_df['Day'] == day) | (selected_df['Day'] <= 0),
            ['Participant_ID', 'Day', metric]
        ].rename(columns={metric: 'value'}).copy()
        day_data['period'] = (day_data['Day'] == day).astype(int)
        row = fit_mixed_model(day_data, outcome='value', predictor='period')
        rows.append({'Metric': metric, 'Day': day, **row})
    daily_tables.append(adjust_fdr(pd.DataFrame(rows)))

daily_results = pd.concat(daily_tables, ignore_index=True)

valid_days = daily_results.dropna(subset=['Adjusted p-value'])
best_days = valid_days.loc[
    valid_days.groupby('Metric')['Adjusted p-value'].idxmin()
].reset_index(drop=True)

significant_days_2_3 = daily_results.loc[
    daily_results['Significant'].fillna(False) & daily_results['Day'].between(2, 3)
].sort_values(['Metric', 'Adjusted p-value']).reset_index(drop=True)

display(best_days.style.format(result_format, na_rep='—'))
display(significant_days_2_3.style.format(result_format, na_rep='—'))

## 3. Test the association between wearable metrics and symptom score

**Question:** During post-inoculation days, is symptom score associated with each wearable metric?

Use post-inoculation data (Day > 0). As in the original code, remove rows missing symptom score or any of the four wearable metrics. Fit `Symptom_Score ~ wearable metric`. BH-FDR is applied across four metrics.

In [ ]:
analysis_symptom = selected_df.loc[
    selected_df['Day'] > 0, ['Participant_ID', 'Symptom_Score'] + metrics
].dropna().reset_index(drop=True)

symptom_results = test_metrics(analysis_symptom, outcome='Symptom_Score')
display(symptom_results.style.format(result_format, na_rep='—'))

## 3.1 Test the symptom-score association after baseline covariants adjustment

**Question:** Is each wearable metric associated with symptom score after adjusting for SEX, AGE, and BMI?

Fit `Symptom_Score ~ wearable metric + SEX + AGE + BMI`. BH-FDR is applied across the four adjusted models, separately from Step 7.

In [ ]:
symptom_adjusted_results = test_metrics(
    add_covariates(analysis_symptom), outcome='Symptom_Score',
    covariates=baseline_covariates
)
display(symptom_adjusted_results.style.format(result_format, na_rep='—'))

## 4. Test the association between wearable metrics and viral load derived from PCR

**Question:** During post-inoculation days with viral load > 0.5 copies/mL, is log10 viral load associated with each wearable metric?

Use Day > 0. Remove rows missing viral load or any of the four wearable metrics, retain viral load > 0.5, and take log10. Fit `viral_load ~ wearable metric`. BH-FDR is applied across four metrics.

In [ ]:
analysis_viral = selected_df.loc[
    selected_df['Day'] > 0, ['Participant_ID', 'Viral Load (copies per mL)'] + metrics
].dropna().reset_index(drop=True)

if (analysis_viral['Viral Load (copies per mL)'] <= 0).any():
    raise ValueError('PCR viral load must be positive before log10; check the prepared input data.')
analysis_viral['viral_load'] = np.log10(analysis_viral['Viral Load (copies per mL)'])

viral_results = test_metrics(analysis_viral, outcome='viral_load')
display(viral_results.style.format(result_format, na_rep='—'))

## 4.2 Test the viral-load association after baseline covariants adjustment

**Question:** Is each wearable metric associated with log10 PCR viral load after adjusting for SEX, AGE, and BMI?

Fit `viral_load ~ wearable metric + SEX + AGE + BMI`. BH-FDR is applied across the four adjusted models, separately from Step 9.

In [ ]:
viral_adjusted_results = test_metrics(
    add_covariates(analysis_viral), outcome='viral_load',
    covariates=baseline_covariates
)
display(viral_adjusted_results.style.format(result_format, na_rep='—'))